## Cell 1: Mount Google Drive
This cell mounts your Google Drive to `/content/drive`, creating a persistent storage location for your downloaded papers and source code. It also sets up base directories for the project.

In [1]:
# ============================================
# Cell 1: Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')

# Create base directory
DRIVE_BASE = '/content/drive/MyDrive/biorag'
!mkdir -p {DRIVE_BASE}/papers
!mkdir -p {DRIVE_BASE}/src

print(f"Drive mounted. Working directory: {DRIVE_BASE}")

Mounted at /content/drive
Drive mounted. Working directory: /content/drive/MyDrive/biorag


## Cell 2: Create `downloader.py`
This cell writes the Python script `downloader.py` directly into your Google Drive at `/content/drive/MyDrive/biorag/src/`. This script contains functions to search PubMed Central (PMC), fetch full-text XMLs, and save them locally.

In [2]:
# ============================================
# Cell 2: Create downloader.py directly in Colab
# ============================================
%%writefile /content/drive/MyDrive/biorag/src/downloader.py
"""
downloader.py - Complete working version for Colab
Fetches open-access full-text XMLs from PubMed Central (PMC)
"""

import os
import time
import requests
from pathlib import Path
from typing import List, Optional

BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def search_pmc(query: str, max_results: int = 30, api_key: str = None) -> List[str]:
    """Search PMC for open-access papers. Returns list of PMC IDs."""
    params = {
        "db": "pmc",
        "term": f"{query} AND open access[filter]",
        "retmax": max_results,
        "retmode": "json",
        "usehistory": "y",
    }
    if api_key:
        params["api_key"] = api_key

    print(f"[Searching] '{query}'...")
    response = requests.get(f"{BASE_URL}/esearch.fcgi", params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    ids = data.get("esearchresult", {}).get("idlist", [])
    print(f"[Found] {len(ids)} papers")
    return ids

def fetch_paper_xml(pmc_id: str, api_key: str = None, delay: float = 0.4) -> Optional[str]:
    """Fetch full-text XML for a single PMC ID."""
    time.sleep(delay)

    params = {
        "db": "pmc",
        "id": pmc_id,
        "rettype": "full",
        "retmode": "xml",
    }
    if api_key:
        params["api_key"] = api_key

    try:
        response = requests.get(f"{BASE_URL}/efetch.fcgi", params=params, timeout=30)
        response.raise_for_status()

        # Check if we got actual content
        if len(response.text) < 500 or "<error" in response.text.lower():
            return None

        return response.text
    except Exception as e:
        print(f"  Error: {e}")
        return None

def save_xml(pmc_id: str, xml_text: str, output_dir: Path) -> Path:
    """Save XML to disk."""
    output_dir.mkdir(parents=True, exist_ok=True)
    filepath = output_dir / f"PMC{pmc_id}.xml"
    filepath.write_text(xml_text, encoding="utf-8")
    return filepath

def download_papers(
    query: str,
    output_dir: str | Path,
    max_results: int = 30,
    api_key: str = None,
) -> List[Path]:
    """Full pipeline: search → fetch → save XMLs."""
    output_dir = Path(output_dir)
    delay = 0.1 if api_key else 0.4

    print(f"\n{'='*50}")
    print(f"Topic: {query}")
    print(f"Output: {output_dir}")
    print(f"{'='*50}\n")

    # Search
    ids = search_pmc(query, max_results, api_key)

    if not ids:
        print("No results found. Try a broader query.")
        return []

    # Download each paper
    saved = []
    for i, pmc_id in enumerate(ids, 1):
        print(f"[{i}/{len(ids)}] PMC{pmc_id}...", end=" ")

        # Check cache
        existing = output_dir / f"PMC{pmc_id}.xml"
        if existing.exists():
            print("cached")
            saved.append(existing)
            continue

        # Download
        xml_text = fetch_paper_xml(pmc_id, api_key, delay)

        if xml_text:
            path = save_xml(pmc_id, xml_text, output_dir)
            saved.append(path)
            size_kb = len(xml_text) // 1000
            print(f"saved ({size_kb}KB)")
        else:
            print("skipped (no full text)")

    print(f"\n[Complete] Saved {len(saved)} / {len(ids)} papers")
    return saved

def list_downloaded(output_dir: str | Path) -> List[Path]:
    """List all downloaded XMLs."""
    output_dir = Path(output_dir)
    if not output_dir.exists():
        return []
    return sorted(output_dir.glob("*.xml"))

print("✅ downloader.py created successfully in your Drive")

Writing /content/drive/MyDrive/biorag/src/downloader.py


## Cell 3: Configure Your Search
In this cell, you define the topic for your search, the maximum number of papers to download, and optionally your NCBI API key. It also dynamically creates the output directory path based on your chosen topic.

In [3]:
# ============================================
# Cell 3: Configure your search
# ============================================
import re

# ---- CONFIGURE HERE ----
TOPIC = 'lung cancer biomarkers'       # Change this to any biomedical topic
MAX_PAPERS = 30                         # Number of papers to search
NCBI_API_KEY = None                     # Optional: get from NCBI account
# ------------------------

# Create output directory on Drive
topic_slug = re.sub(r'[^a-z0-9]+', '_', TOPIC.lower()).strip('_')
OUTPUT_DIR = f'{DRIVE_BASE}/papers/{topic_slug}'

print(f"Topic       : {TOPIC}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Max papers  : {MAX_PAPERS}")
print(f"API key     : {'Set' if NCBI_API_KEY else 'Not set (3 req/sec)'}")

Topic       : lung cancer biomarkers
Output dir  : /content/drive/MyDrive/biorag/papers/lung_cancer_biomarkers
Max papers  : 30
API key     : Not set (3 req/sec)


## Cell 4: Add Source to Path and Import
This cell adds the `src` directory (where `downloader.py` resides) to the system's Python path, allowing you to import functions from the `downloader.py` script.

In [4]:
# ============================================
# Cell 4: Add src to path and import
# ============================================
import sys
sys.path.insert(0, f'{DRIVE_BASE}/src')

# Import the downloader
from downloader import download_papers, list_downloaded

print("✅ Imports successful")

✅ downloader.py created successfully in your Drive
✅ Imports successful


## Cell 5: Download Papers
This is the core execution cell. It calls the `download_papers` function with your configured `TOPIC`, `OUTPUT_DIR`, `MAX_PAPERS`, and `NCBI_API_KEY` to perform the search, download, and saving of XML files.

In [5]:
# ============================================
# Cell 5: Download papers
# ============================================
saved_files = download_papers(
    query=TOPIC,
    output_dir=OUTPUT_DIR,
    max_results=MAX_PAPERS,
    api_key=NCBI_API_KEY,
)

print(f"\n📊 Summary: Downloaded {len(saved_files)} papers")


Topic: lung cancer biomarkers
Output: /content/drive/MyDrive/biorag/papers/lung_cancer_biomarkers

[Searching] 'lung cancer biomarkers'...
[Found] 30 papers
[1/30] PMC13135549... saved (174KB)
[2/30] PMC13135499... saved (185KB)
[3/30] PMC13135614... saved (158KB)
[4/30] PMC13135508... saved (52KB)
[5/30] PMC13135767... saved (137KB)
[6/30] PMC13135628... saved (193KB)
[7/30] PMC13135828... saved (107KB)
[8/30] PMC13135751... saved (300KB)
[9/30] PMC13135692... saved (291KB)
[10/30] PMC13135784... saved (83KB)
[11/30] PMC13135473... saved (128KB)
[12/30] PMC13135792... saved (236KB)
[13/30] PMC13135812... saved (84KB)
[14/30] PMC13135610... saved (157KB)
[15/30] PMC13135589... saved (480KB)
[16/30] PMC13135612... saved (212KB)
[17/30] PMC13135539... saved (94KB)
[18/30] PMC13135722... saved (183KB)
[19/30] PMC13135720... saved (107KB)
[20/30] PMC13135662... saved (64KB)
[21/30] PMC13135512... saved (158KB)
[22/30] PMC13135236... saved (337KB)
[23/30] PMC13135135... saved (122KB)
[24/3

## Cell 6: Quick Sanity Check - Peek at an XML
This cell performs a quick check to ensure that the downloaded XML files are valid and contain expected elements like titles and abstracts. It uses `ElementTree` to parse the XML and prints a snippet of information from the first downloaded file.

In [8]:
# ============================================
# Cell 7: Quick sanity check - peek at one XML (FIXED)
# ============================================
import xml.etree.ElementTree as ET
from pathlib import Path

# Get files fresh from the directory
files = list_downloaded(OUTPUT_DIR)

if files:
    sample = files[0]
    try:
        tree = ET.parse(sample)
        root = tree.getroot()

        # Try to find title
        title_el = root.find('.//article-title')
        abstract_el = root.find('.//abstract')

        print(f"✅ File: {sample.name}")
        print(f"   Root tag: {root.tag}")
        if title_el is not None and title_el.text:
            title = title_el.text[:100] + "..." if len(title_el.text) > 100 else title_el.text
            print(f"   Title: {title}")
        else:
            print("   Title: not found")
        print(f"   Has abstract: {abstract_el is not None}")

        # Also show a snippet of text if available
        if abstract_el is not None:
            abstract_text = ''.join(abstract_el.itertext())[:200]
            if abstract_text:
                print(f"\n   Abstract snippet: {abstract_text}...")
    except Exception as e:
        print(f"⚠️  Error parsing XML: {e}")
else:
    print("❌ No files found. Run Cell 5 first to download papers.")

✅ File: PMC13134945.xml
   Root tag: pmc-articleset
   Title: Efficacy of Docetaxel Plus Ramucirumab for Malignant Pleural Effusion and Cerebral Edema in Patients...
   Has abstract: True

   Abstract snippet: ABSTRACTBackgroundMalignant pleural effusion (MPE) is associated with a poor prognosis and quality of life in patients with non‐small cell lung cancer (NSCLC). Additionally, cerebral edema can lead to...


## Cell 7: Display File Paths for Next Notebook
This cell simply prints the path to the directory where the papers were saved. This path (`PAPER_DIR`) is crucial for linking this notebook's output to subsequent notebooks in the project workflow.

In [9]:
# ============================================
# Cell 8: Display file paths for next notebook
# ============================================
print("✅ Papers saved to:")
print(f"   {OUTPUT_DIR}")
print(f"\n📌 For Notebook 02, use:")
print(f"   PAPER_DIR = '{OUTPUT_DIR}'")

✅ Papers saved to:
   /content/drive/MyDrive/biorag/papers/lung_cancer_biomarkers

📌 For Notebook 02, use:
   PAPER_DIR = '/content/drive/MyDrive/biorag/papers/lung_cancer_biomarkers'
